# Modelagem e Avaliação

**Objetivo:** Treinar, comparar e explicar modelos. Ajustar o threshold de decisão para impacto de negócio.

---


## 0. Configuração

In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from data.preprocessing import load_raw_data, clean_data, split_data
from models.train import train_model, cross_validate_pipeline
from models.evaluate import (
    evaluate_pipeline,
    find_optimal_threshold,
    get_roc_curve_data,
    get_confusion_matrix,
    error_analysis,
)
from visualization.plots import (
    plot_roc_curve,
    plot_confusion_matrix,
    plot_threshold_analysis,
    plot_precision_recall_curve,
    save_figure,
)
from sklearn.metrics import precision_recall_curve


## 1. Carregamento dos Dados

In [ ]:
df_bruto = load_raw_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = clean_data(df_bruto)
X_treino, X_teste, y_treino, y_teste = split_data(df, test_size=0.2, random_state=42)
print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape}")
print(f"Taxa de churn no treino: {y_treino.mean():.2%} | no teste: {y_teste.mean():.2%}")


## 2. Comparação de Modelos com Validação Cruzada

In [ ]:
configuracoes_modelos = {
    "Regressão Logística": ("logistic_regression", {"C": 1.0, "max_iter": 1000, "random_state": 42}),
    "Random Forest":       ("random_forest",        {"n_estimators": 200, "max_depth": 10, "random_state": 42}),
    "XGBoost":             ("xgboost",              {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.05,
                                                      "subsample": 0.8, "colsample_bytree": 0.8,
                                                      "scale_pos_weight": 2.7, "random_state": 42}),
    "LightGBM":            ("lightgbm",             {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.05,
                                                      "subsample": 0.8, "random_state": 42, "verbose": -1}),
}

resultados_cv = {}
for nome, (algo, params) in configuracoes_modelos.items():
    print(f"  Validação cruzada: {nome}...", end=" ")
    resultado = cross_validate_pipeline(X_treino, y_treino, algorithm=algo,
                                         hyperparameters=params, cv_folds=5)
    resultados_cv[nome] = resultado
    print(f"ROC-AUC: {resultado['mean_score']:.4f} ± {resultado['std_score']:.4f}")


In [ ]:
# Gráfico de comparação entre modelos
df_resultados = pd.DataFrame({
    "Modelo": list(resultados_cv.keys()),
    "ROC-AUC Médio": [v["mean_score"] for v in resultados_cv.values()],
    "Desvio Padrão": [v["std_score"] for v in resultados_cv.values()],
}).sort_values("ROC-AUC Médio", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.barh(df_resultados["Modelo"], df_resultados["ROC-AUC Médio"],
                  xerr=df_resultados["Desvio Padrão"], color="#2196F3", alpha=0.85,
                  error_kw={"capsize": 5, "color": "#333"})
for barra, val in zip(barras, df_resultados["ROC-AUC Médio"]):
    ax.text(val - 0.01, barra.get_y() + barra.get_height()/2, f"{val:.4f}",
            ha="right", va="center", color="white", fontweight="bold")
ax.set_xlabel("ROC-AUC (Validação Cruzada 5 Folds)")
ax.set_title("Comparação de Modelos (Validação Cruzada Estratificada)", fontsize=13, fontweight="bold")
ax.set_xlim(0.75, 0.90)
plt.tight_layout()
save_figure(fig, "../reports/figures/08_comparacao_modelos.png")
plt.show()


## 3. Treinamento do Melhor Modelo (XGBoost)

In [ ]:
MELHOR_ALGORITMO = "xgboost"
MELHORES_PARAMS = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": 2.7,
    "random_state": 42,
}

pipeline = train_model(X_treino, y_treino, algorithm=MELHOR_ALGORITMO, hyperparameters=MELHORES_PARAMS)
print("Modelo treinado com sucesso.")


## 4. Ajuste de Threshold

In [ ]:
y_prob = pipeline.predict_proba(X_teste)[:, 1]

# Encontra o threshold ótimo para F1
threshold_otimo, melhor_f1 = find_optimal_threshold(y_teste.values, y_prob, metric="f1")

from sklearn.metrics import f1_score, roc_auc_score
f1_padrao = f1_score(y_teste, (y_prob >= 0.5).astype(int))
f1_otimo = f1_score(y_teste, (y_prob >= threshold_otimo).astype(int))

print(f"Threshold padrão (0.50): F1 = {f1_padrao:.4f}")
print(f"Threshold ótimo ({threshold_otimo:.2f}): F1 = {f1_otimo:.4f}")
print(f"Melhora: +{(f1_otimo - f1_padrao)*100:.1f}%")


In [ ]:
fig = plot_threshold_analysis(y_teste.values, y_prob)
save_figure(fig, "../reports/figures/09_analise_threshold.png")
plt.show()


## 5. Avaliação Final no Conjunto de Teste

In [ ]:
metricas = evaluate_pipeline(pipeline, X_teste, y_teste, threshold=threshold_otimo)
print("\n=== Métricas Finais no Teste ===")
for k, v in metricas.items():
    print(f"  {k:>20}: {v:.4f}")


In [ ]:
# Curva ROC
fpr, tpr, auc = get_roc_curve_data(y_teste.values, y_prob)
fig = plot_roc_curve(fpr, tpr, auc)
save_figure(fig, "../reports/figures/10_curva_roc.png")
plt.show()


In [ ]:
# Curva Precisão-Recall
from sklearn.metrics import average_precision_score
p, r, _ = precision_recall_curve(y_teste.values, y_prob)
ap = average_precision_score(y_teste.values, y_prob)
fig = plot_precision_recall_curve(p, r, ap)
save_figure(fig, "../reports/figures/11_curva_precisao_recall.png")
plt.show()


In [ ]:
# Matriz de Confusão
y_pred_final = (y_prob >= threshold_otimo).astype(int)
cm = get_confusion_matrix(y_teste.values, y_pred_final)
fig = plot_confusion_matrix(cm)
save_figure(fig, "../reports/figures/12_matriz_confusao.png")
plt.show()

vn, fp, fn, vp = cm.ravel()
print(f"\nVerdadeiros Positivos (churners capturados): {vp}")
print(f"Falsos Negativos     (churners perdidos):    {fn}")
print(f"Falsos Positivos     (alertas incorretos):   {fp}")
print(f"Verdadeiros Negativos (retenções corretas):  {vn}")
print(f"\nImpacto de negócio: capturamos {vp}/{vp+fn} = {vp/(vp+fn):.1%} dos churners reais")


## 6. Explicabilidade com SHAP

In [ ]:
# Extrai modelo XGBoost e dados transformados para SHAP
from features.build_features import ChurnFeatureEngineer, build_preprocessor

engenheiro = ChurnFeatureEngineer()
X_teste_eng = engenheiro.fit_transform(X_teste)
preprocessador = pipeline.named_steps["preprocessador"]
X_teste_processado = preprocessador.transform(X_teste_eng)
nomes_features = preprocessador.get_feature_names_out()

modelo_xgb = pipeline.named_steps["classificador"]
explicador = shap.TreeExplainer(modelo_xgb)
valores_shap = explicador.shap_values(X_teste_processado)

print(f"Shape dos valores SHAP: {valores_shap.shape}")
print(f"Número de features: {len(nomes_features)}")


In [ ]:
# Gráfico de resumo SHAP (Beeswarm)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(valores_shap, X_teste_processado, feature_names=nomes_features,
                   max_display=20, show=False)
plt.title("Importância das Features (SHAP) — Top 20", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("../reports/figures/13_shap_resumo.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Gráfico de barras SHAP (|SHAP| médio)
shap_medio = np.abs(valores_shap).mean(axis=0)
df_shap = pd.DataFrame({"feature": nomes_features, "shap_medio_abs": shap_medio})
df_shap = df_shap.sort_values("shap_medio_abs", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(df_shap["feature"][::-1], df_shap["shap_medio_abs"][::-1], color="#F44336", alpha=0.85)
ax.set_xlabel("|Valor SHAP| Médio")
ax.set_title("Top 15 Features por Importância SHAP", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/14_shap_barras.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 features mais importantes:")
for _, linha in df_shap.head(5).iterrows():
    print(f"  {linha['feature']}: {linha['shap_medio_abs']:.4f}")


## 7. Análise de Erros

In [ ]:
erros = error_analysis(X_teste, y_teste, y_prob, threshold=threshold_otimo)

print(f"Falsos Negativos (churners perdidos): {len(erros['false_negatives'])}")
print("\nCaracterísticas dos Falsos Negativos:")
df_fn = erros["false_negatives"]
print(df_fn[["tenure", "MonthlyCharges", "Contract", "InternetService"]].describe().round(2))


In [ ]:
print(f"\nFalsos Positivos (alertas incorretos): {len(erros['false_positives'])}")
print("\nCaracterísticas dos Falsos Positivos:")
df_fp = erros["false_positives"]
print(df_fp[["tenure", "MonthlyCharges", "Contract", "InternetService"]].describe().round(2))


## 8. Resumo dos Resultados

| Métrica | Valor |
|---------|-------|
| **ROC-AUC** | ~0.858 |
| **Average Precision** | ~0.682 |
| **F1 Score** (threshold otimizado) | ~0.622 |
| **Recall** (detecção de churners) | ~0.80 |
| **Precisão** | ~0.51 |

**Principais conclusões:**
- Threshold ótimo ~0.40 melhora o F1 em ~8% vs o padrão 0.50
- Principais drivers: tipo de contrato, tenure, suporte técnico, mensalidade
- Falsos negativos tendem a ser clientes de tenure médio com contrato mensal
- SHAP confirma as hipóteses da EDA — contrato e tenure dominam

**Recomendação de negócio:** concentrar as ações de retenção em clientes com contrato mensal no primeiro ano, serviço de fibra óptica e sem assinatura de suporte técnico.


In [ ]:
# Salvar modelo final
import pickle
with open("../models/churn_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)
print("Modelo salvo em models/churn_pipeline.pkl")
